# Chapter 15 — Agents Are Programs Too

**Book alignment:** DSPy From First Principles, Chapter 15

**Question this notebook isolates:** Does a bounded read-only tool surface block a path-escape probe while still returning the evidence a diagnosis needs?


In [ ]:
from pathlib import Path
import inspect
import sys
import tempfile


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy
from common.repository_tools import RepositoryTools


class DiagnoseRepositoryIssue(dspy.Signature):
    """Diagnose a repository issue using only the supplied read-only tools."""

    issue: str = dspy.InputField()
    constraints: str = dspy.InputField()

    diagnosis: str = dspy.OutputField()
    suspected_files: list[str] = dspy.OutputField()
    missing_evidence: str = dspy.OutputField()


def _search_repository(query: str) -> str:
    """Placeholder matching the declared tool shape; never executed against a model."""
    raise NotImplementedError("no-LM notebook: tool shape only")


def _read_file(path: str) -> str:
    raise NotImplementedError("no-LM notebook: tool shape only")


def _inspect_symbol(name: str) -> str:
    raise NotImplementedError("no-LM notebook: tool shape only")


DECLARED_TOOLS = {"search_repository", "read_file", "inspect_symbol"}
agent = dspy.ReAct(
    DiagnoseRepositoryIssue,
    tools=[_search_repository, _read_file, _inspect_symbol],
    max_iters=6,
)
print("dspy", dspy.__version__, "| ReAct constructed, never executed")


## Bounded tools return the needed evidence

A two-file fixture mirrors the book: `models.py` defines `Item.price`, `service.py` sums `item.cost`. The tools are bound to the fixture root in application code and read through `RepositoryTools` only.


In [ ]:
_tmp = tempfile.TemporaryDirectory()
FIXTURE = Path(_tmp.name) / "checkout"
FIXTURE.mkdir()
(FIXTURE / "models.py").write_text(
    "class Item:\n    def __init__(self, name, price):\n        self.name = name\n        self.price = price\n",
    encoding="utf-8",
)
(FIXTURE / "service.py").write_text(
    "from models import Item\n\n\ndef total(items):\n    return sum(item.cost for item in items)\n",
    encoding="utf-8",
)
SENTINEL = Path(_tmp.name) / "outside.txt"
SENTINEL.write_text("secret outside the repository root\n", encoding="utf-8")

tools = RepositoryTools(root=FIXTURE)
search_res = tools.search_repository("cost")
read_res = tools.read_file("service.py", 1, 40)
sym_res = tools.inspect_symbol("Item")

search_paths = [m["path"] for m in search_res.data] if search_res.ok else []
read_text = "\n".join(l["text"] for l in read_res.data["lines"]) if read_res.ok else ""
sym_hits = [(h["path"], h["kind"], h["line"]) for h in sym_res.data] if sym_res.ok else []
print("search paths:", search_paths)
print("read ok:", read_res.ok, "| chars:", len(read_text))
print("symbol hits:", sym_hits)


In [ ]:
assert search_res.ok and "service.py" in search_paths
assert read_res.ok and "item.cost" in read_text
assert sym_res.ok and any(p == "models.py" for p, _, _ in sym_hits)
assert "outside" not in str(search_paths) + read_text + str(sym_hits)
print("evidence present; outside sentinel never observed")


## Attack the boundary you just built

Three checks from the chapter: a path-escape probe is blocked, the root is absent from every model-callable signature, and the runtime surface equals the declared surface.


In [ ]:
escape = tools.read_file("../outside.txt")
sigs = {
    name: set(inspect.signature(fn).parameters)
    for name, fn in [
        ("search_repository", tools.search_repository),
        ("read_file", tools.read_file),
        ("inspect_symbol", tools.inspect_symbol),
    ]
}
actual_surface = {fn.__name__ for fn in [tools.search_repository, tools.read_file, tools.inspect_symbol]}
print("escape ok:", escape.ok, "| error:", escape.error)
print("signature params:", {k: sorted(v) for k, v in sigs.items()})
print("surface:", sorted(actual_surface))


In [ ]:
assert escape.ok is False and "escapes repository root" in str(escape.error)
assert all("root" not in params for params in sigs.values())
assert actual_surface == DECLARED_TOOLS
print("boundary held: escape blocked, root application-bound, surface declared")


## Right evidence, wrong answer

The book's measured run held both halves of the defect in context and still restated the traceback. The runner's decomposed score is reproduced here as arithmetic: the failure is in one named component, not in the evidence.


In [ ]:
components = [
    {"name": "root_cause_correct", "weight": 0.40, "passed": False},
    {"name": "suspected_file_correct", "weight": 0.25, "passed": True},
    {"name": "relevant_evidence_inspected", "weight": 0.20, "passed": True},
    {"name": "valid_tool_use", "weight": 0.15, "passed": True},
]
for comp in components:
    comp["contribution"] = comp["weight"] if comp["passed"] else 0.0
total = sum(comp["contribution"] for comp in components)
for comp in components:
    print(f"{comp['name']:28s} weight={comp['weight']:.2f} passed={comp['passed']!s:5s} -> {comp['contribution']:.2f}")
print(f"{'total':28s} {'':22s} -> {total:.2f}")


In [ ]:
assert abs(total - 0.60) < 1e-9
assert components[0]["passed"] is False
assert all(comp["passed"] for comp in components[1:])
print("0.60 = full marks everywhere except the 0.40 root-cause component")


## What we earned

Safety and correctness separated cleanly: the boundary blocked the escape probe while the tools returned both implicated files, and the decomposed score locates the failure in reasoning rather than access.

Notebook 16 / Chapter 16 keeps the bounded program fixed and varies how supplemental evidence is selected before the model runs.
